In [7]:
import torch
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder

# Choosing Apple MPS for optimization
device = "mps"

# Loading Bi-Encoder
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# Loading Cross-Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3704.90it/s]
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10943.02it/s]


In [12]:
# Practice dataset pairs to test if a semantic cache is vulnerable to cache poisioning. The pairs are split into positive pairs that equal 1 and negative pairs that equal 0.

dataset = [
    # Positive
    {
        "prompt_a": "How do I configure Redis connection pooling in Python?",
        "prompt_b": "Show me how to set up a Redis connection pool using Python.",
        "label": 1,
        "category": "paraphrase"
    },
    {
        "prompt_a": "Write a SQL query to find the top 5 highest paid employees.",
        "prompt_b": "Give me a SQL query returning the top 5 employees with the highest salary.",
        "label": 1,
        "category": "paraphrase"
    },
    {
        "prompt_a": "Explain the difference between supervised and unsupervised learning.",
        "prompt_b": "What is the distinction between supervised vs unsupervised machine learning?",
        "label": 1,
        "category": "paraphrase"
    },
    {
        "prompt_a": "Generate a FastAPI endpoint to upload PDF files.",
        "prompt_b": "Create a route in FastAPI that accepts PDF file uploads.",
        "label": 1,
        "category": "paraphrase"
    },

    # Negative
    {
        "prompt_a": "Drop the production database table 'users'.",
        "prompt_b": "Do NOT drop the production database table 'users'.",
        "label": 0,
        "category": "negation"
    },
    {
        "prompt_a": "Grant full administrative privileges to user id 402.",
        "prompt_b": "Revoke all administrative privileges from user id 402.",
        "label": 0,
        "category": "antonym_action"
    },
    {
        "prompt_a": "Transfer $1,000 from account checking_1 to account savings_2.",
        "prompt_b": "Transfer $1,000 from account savings_2 to account checking_1.",
        "label": 0,
        "category": "parameter_swap"
    },
    {
        "prompt_a": "Sort the user transaction history in ascending order of timestamp.",
        "prompt_b": "Sort the user transaction history in descending order of timestamp.",
        "label": 0,
        "category": "order_flip"
    },
    {
        "prompt_a": "Filter rows where user age is strictly greater than 18.",
        "prompt_b": "Filter rows where user age is strictly less than 18.",
        "label": 0,
        "category": "logic_flip"
    },
    {
        "prompt_a": "Enable debug logging and disable TLS certificate verification in production.",
        "prompt_b": "Disable debug logging and enable TLS certificate verification in production.",
        "label": 0,
        "category": "multi_flag_flip"
    }
]

# View as a structured dataframe
df = pd.DataFrame(dataset)
print(f"Total test pairs: {len(df)}")
print(f"Positive pairs: {(df['label'] == 1).sum()}")
print(f"Negative pairs: {(df['label'] == 0).sum()}")

# Viewing top values
df.head()

Total test pairs: 10
Positive pairs: 4
Negative pairs: 6


,prompt_a,prompt_b,label,category
0,How do I configure Redis connection pooling in...,Show me how to set up a Redis connection pool ...,1,paraphrase
1,Write a SQL query to find the top 5 highest pa...,Give me a SQL query returning the top 5 employ...,1,paraphrase
2,Explain the difference between supervised and ...,What is the distinction between supervised vs ...,1,paraphrase
3,Generate a FastAPI endpoint to upload PDF files.,Create a route in FastAPI that accepts PDF fil...,1,paraphrase
4,Drop the production database table 'users'.,Do NOT drop the production database table 'use...,0,negation


In [17]:
# Encode both lists of prompts using the bi-encoder and normalize
# By normalizing we simplify the cosine simplarity to a dot product
embeddings_a = bi_encoder.encode(df["prompt_a"].tolist(), convert_to_numpy=True, normalize_embeddings=True)
embeddings_b = bi_encoder.encode(df["prompt_b"].tolist(), convert_to_numpy=True, normalize_embeddings=True)

# Calculate the row-wise aka the embeddings a vs embeddings b and does it across the 384 layer dimensions
df["bi_sim"] = np.sum(embeddings_a * embeddings_b, axis=1)

# Choose decision threshold for the comparisons
tau_bi = 0.85
df["bi_cache_hit"] = df["bi_sim"] >= tau_bi

# Find false posistives aka when label is 0 but cache hit is true
df["cache_poisoned"] = (df["label"] == 0) & (df["bi_cache_hit"] == True)

# Display results
results_view = df[["category", "label", "bi_sim", "bi_cache_hit", "cache_poisoned", "prompt_a", "prompt_b"]]
results_view

,category,label,bi_sim,bi_cache_hit,cache_poisoned,prompt_a,prompt_b
0,paraphrase,1,0.930067,True,False,How do I configure Redis connection pooling in...,Show me how to set up a Redis connection pool ...
1,paraphrase,1,0.932045,True,False,Write a SQL query to find the top 5 highest pa...,Give me a SQL query returning the top 5 employ...
2,paraphrase,1,0.912703,True,False,Explain the difference between supervised and ...,What is the distinction between supervised vs ...
3,paraphrase,1,0.872250,True,False,Generate a FastAPI endpoint to upload PDF files.,Create a route in FastAPI that accepts PDF fil...
4,negation,0,0.946733,True,True,Drop the production database table 'users'.,Do NOT drop the production database table 'use...
5,antonym_action,0,0.802348,False,False,Grant full administrative privileges to user i...,Revoke all administrative privileges from user...
6,parameter_swap,0,0.997791,True,True,"Transfer $1,000 from account checking_1 to acc...","Transfer $1,000 from account savings_2 to acco..."
7,order_flip,0,0.993011,True,True,Sort the user transaction history in ascending...,Sort the user transaction history in descendin...
8,logic_flip,0,0.991008,True,True,Filter rows where user age is strictly greater...,Filter rows where user age is strictly less th...
9,multi_flag_flip,0,0.987286,True,True,Enable debug logging and disable TLS certifica...,Disable debug logging and enable TLS certifica...


In [19]:
# Format prompt pairs for Cross-Encoder
pairs = df[["prompt_a", "prompt_b"]].values.tolist()

# Eval pairs raw logits
raw_ce_scores = cross_encoder.predict(pairs)
df["ce_raw_score"] = raw_ce_scores

# Apply Sigmoid function
df["ce_prob"] = 1.0 / (1.0 + np.exp(-raw_ce_scores))

# Compare the Bi-Encoder and Cross-Encoder
comparison_view = df[[
    "category", 
    "label", 
    "bi_sim", 
    "ce_raw_score", 
    "ce_prob", 
    "prompt_a", 
    "prompt_b"
]]
comparison_view

,category,label,bi_sim,ce_raw_score,ce_prob,prompt_a,prompt_b
0,paraphrase,1,0.930067,4.361175,0.987397,How do I configure Redis connection pooling in...,Show me how to set up a Redis connection pool ...
1,paraphrase,1,0.932045,7.677634,0.999537,Write a SQL query to find the top 5 highest pa...,Give me a SQL query returning the top 5 employ...
2,paraphrase,1,0.912703,4.658004,0.990604,Explain the difference between supervised and ...,What is the distinction between supervised vs ...
3,paraphrase,1,0.872250,5.683901,0.996611,Generate a FastAPI endpoint to upload PDF files.,Create a route in FastAPI that accepts PDF fil...
4,negation,0,0.946733,8.222992,0.999732,Drop the production database table 'users'.,Do NOT drop the production database table 'use...
5,antonym_action,0,0.802348,5.600006,0.996316,Grant full administrative privileges to user i...,Revoke all administrative privileges from user...
6,parameter_swap,0,0.997791,9.007404,0.999877,"Transfer $1,000 from account checking_1 to acc...","Transfer $1,000 from account savings_2 to acco..."
7,order_flip,0,0.993011,8.424584,0.999781,Sort the user transaction history in ascending...,Sort the user transaction history in descendin...
8,logic_flip,0,0.991008,9.178064,0.999897,Filter rows where user age is strictly greater...,Filter rows where user age is strictly less th...
9,multi_flag_flip,0,0.987286,9.419815,0.999919,Enable debug logging and disable TLS certifica...,Disable debug logging and enable TLS certifica...


In [21]:
# Separate the dataset into Positives and Negatives
positives = df[df["label"] == 1]
negatives = df[df["label"] == 0]

# Measure the accuracy of Bi-Encoder at tau = 0.85
bi_true_positives = (positives["bi_sim"] >= tau_bi).sum()
bi_false_positives = (negatives["bi_sim"] >= tau_bi).sum()

bi_hit_rate_on_paraphrases = bi_true_positives / len(positives)
bi_cache_poisoning_rate = bi_false_positives / len(negatives)

print("LAB 1 EXPERIMENTAL RESULTS: BI-ENCODER CACHE EVALUATION")
print(f"Total Test Pairs: {len(df)}")
print(f"Valid Paraphrase Cache Hit Rate (Recall):  {bi_hit_rate_on_paraphrases * 100:.1f}% ({bi_true_positives}/{len(positives)})")
print(f"Adversarial Cache Poisoning Rate (FPR):    {bi_cache_poisoning_rate * 100:.1f}% ({bi_false_positives}/{len(negatives)})")

# Output of failures and breakdown
print("\nAdversarial Pairs that Bypassed the Cache (Poisoning Events):")
for _, row in negatives[negatives["bi_sim"] >= 0.85].iterrows():
    print(f"- [{row['category'].upper()}] (Sim: {row['bi_sim']:.3f})")
    print(f"    Cached:   \"{row['prompt_a']}\"")
    print(f"    Incoming: \"{row['prompt_b']}\"")

LAB 1 EXPERIMENTAL RESULTS: BI-ENCODER CACHE EVALUATION
Total Test Pairs: 10
Valid Paraphrase Cache Hit Rate (Recall):  100.0% (4/4)
Adversarial Cache Poisoning Rate (FPR):    83.3% (5/6)

Adversarial Pairs that Bypassed the Cache (Poisoning Events):
- [NEGATION] (Sim: 0.947)
    Cached:   "Drop the production database table 'users'."
    Incoming: "Do NOT drop the production database table 'users'."
- [PARAMETER_SWAP] (Sim: 0.998)
    Cached:   "Transfer $1,000 from account checking_1 to account savings_2."
    Incoming: "Transfer $1,000 from account savings_2 to account checking_1."
- [ORDER_FLIP] (Sim: 0.993)
    Cached:   "Sort the user transaction history in ascending order of timestamp."
    Incoming: "Sort the user transaction history in descending order of timestamp."
- [LOGIC_FLIP] (Sim: 0.991)
    Cached:   "Filter rows where user age is strictly greater than 18."
    Incoming: "Filter rows where user age is strictly less than 18."
- [MULTI_FLAG_FLIP] (Sim: 0.987)
    Cached:

In [22]:
# Evaluate cache poisioning behavior across multiple threshold values
thresholds = [0.75, 0.80, 0.85, 0.90, 0.92, 0.95, 0.98]
sweep_results = []

for tau in thresholds:
	valid_hits = (positives["bi_sim"] >= tau).sum() / len(positives)
	poison_hits = (negatives["bi_sim"] >= tau).sum() / len(negatives)

	sweep_results.append({
		"Threshold (tau)": tau,
		"Valid Hit Rate (Recall)": f"{valid_hits *100:.1f}%",
		"Poisoning Rate (FPR)": f"{poison_hits * 100:.1f}%",
		"Safe?": "NO" if poison_hits > 0 else "YES"
	})

sweep_df = pd.DataFrame(sweep_results)
sweep_df

,Threshold (tau),Valid Hit Rate (Recall),Poisoning Rate (FPR),Safe?
0,0.75,100.0%,100.0%,NO
1,0.80,100.0%,100.0%,NO
2,0.85,100.0%,83.3%,NO
3,0.90,75.0%,83.3%,NO
4,0.92,50.0%,83.3%,NO
5,0.95,0.0%,66.7%,NO
6,0.98,0.0%,66.7%,NO
